In [21]:
"안녕하세요 👋 (hello in Korean!)"

'안녕하세요 👋 (hello in Korean!)'

In [22]:
[ord(x)for x in "안녕하세요 👋 (hello in Korean!)"]

[50504,
 45397,
 54616,
 49464,
 50836,
 32,
 128075,
 32,
 40,
 104,
 101,
 108,
 108,
 111,
 32,
 105,
 110,
 32,
 75,
 111,
 114,
 101,
 97,
 110,
 33,
 41]

In [23]:
list('rahul'.encode('utf-8'))

[114, 97, 104, 117, 108]

In [24]:
[ord(c) for c in 'rahul']

[114, 97, 104, 117, 108]

In [25]:
ord('안')

50504

In [26]:
list('안'.encode('utf-8'))

[236, 149, 136]

In [27]:
bytes([236, 149, 136]).decode("utf-8")

'안'

In [28]:
b1, b2, b3 = [236, 149, 136]

codepoint = ((b1 & 0b00001111) << 12) | \
            ((b2 & 0b00111111) << 6)  | \
            (b3 & 0b00111111)

print(codepoint)
print(chr(codepoint))

50504
안


In [29]:
list('👋'.encode('utf-8'))

[240, 159, 145, 139]

In [30]:
[bin(v) for v in [240, 159, 145, 139]]

['0b11110000', '0b10011111', '0b10010001', '0b10001011']

In [31]:
b1, b2, b3, b4 = [240, 159, 145, 139]

codepoint = ((b1 & 0b00000111) << 18) | \
            ((b2 & 0b00111111) << 12) | \
            ((b3 & 0b00111111) << 6)  | \
            (b4 & 0b00111111)

print(codepoint)
print(chr(codepoint))

128075
👋


In [32]:
def encode(s: str):
    return list(s.encode('utf-8'))
def decode(l: list[int]):
    return bytes(l).decode('utf-8')

In [33]:
encode('rahul')

[114, 97, 104, 117, 108]

In [34]:
decode(encode('rahul'))

'rahul'

In [35]:
decode(encode('ਇਕ ਓਅੰਕਾਰ'))

'ਇਕ ਓਅੰਕਾਰ'

In [36]:
encode('ਇ')

[224, 168, 135]

UTF-8 bytes carry their own “shape”, so the decoder can tell how many bytes belong together.

For your Punjabi character:

```python
encode('ਇ')
# [224, 168, 135]
```

Look at those bytes in binary:

```text
224 = 11100000
168 = 10101000
135 = 10000111
```

UTF-8 has prefix rules:

```text
0xxxxxxx  -> 1-byte character
110xxxxx  -> start of 2-byte character
1110xxxx  -> start of 3-byte character
11110xxx  -> start of 4-byte character
10xxxxxx  -> continuation byte
```

Now apply that to your bytes:

```text
224 = 11100000
      ^^^^
      1110 means: start of a 3-byte character

168 = 10101000
      ^^
      10 means: continuation byte

135 = 10000111
      ^^
      10 means: continuation byte
```

So when the decoder sees `224`, it knows:

```text
This character needs 3 bytes total.
Take this byte plus the next 2 continuation bytes.
```

That is why these go together:

```text
[224, 168, 135]
```

Then it removes the UTF-8 marker bits and keeps only the useful bits:

```text
224: 11100000 -> useful bits: 0000
168: 10101000 -> useful bits: 101000
135: 10000111 -> useful bits: 000111
```

Put them together:

```text
0000 101000 000111
```

That binary number is:

```python
((224 & 0b00001111) << 12) | \
((168 & 0b00111111) << 6)  | \
(135 & 0b00111111)
# 2567
```

And:

```python
chr(2567)
# 'ਇ'
```

So the decoder knows because the first byte says the length:

```text
224 starts with 1110 -> this is a 3-byte UTF-8 sequence
```

Then the next two bytes start with `10`, confirming they are continuation bytes.


In [37]:
def custom_encode(s: str) -> list[int]:
    out = []

    for ch in s:
        code = ord(ch)

        if code <= 0x7F:
            # 0xxxxxxx
            out.append(code)

        elif code <= 0x7FF:
            # 110xxxxx 10xxxxxx
            out.append(0b11000000 | (code >> 6))
            out.append(0b10000000 | (code & 0b00111111))

        elif code <= 0xFFFF:
            # 1110xxxx 10xxxxxx 10xxxxxx
            out.append(0b11100000 | (code >> 12))
            out.append(0b10000000 | ((code >> 6) & 0b00111111))
            out.append(0b10000000 | (code & 0b00111111))

        elif code <= 0x10FFFF:
            # 11110xxx 10xxxxxx 10xxxxxx 10xxxxxx
            out.append(0b11110000 | (code >> 18))
            out.append(0b10000000 | ((code >> 12) & 0b00111111))
            out.append(0b10000000 | ((code >> 6) & 0b00111111))
            out.append(0b10000000 | (code & 0b00111111))

        else:
            raise ValueError(f"Invalid Unicode code point: {code}")

    return out


def custom_decode(byte_list: list[int]) -> str:
    chars = []
    i = 0

    def continuation_byte(j: int) -> int:
        if j >= len(byte_list):
            raise ValueError("Unexpected end of bytes")

        b = byte_list[j]
        if b & 0b11000000 != 0b10000000:
            raise ValueError(f"Expected continuation byte at index {j}, got {b}")

        return b & 0b00111111

    while i < len(byte_list):
        b1 = byte_list[i]

        if not 0 <= b1 <= 255:
            raise ValueError(f"Byte out of range at index {i}: {b1}")

        if b1 & 0b10000000 == 0:
            # 0xxxxxxx
            code = b1
            i += 1

        elif b1 & 0b11100000 == 0b11000000:
            # 110xxxxx 10xxxxxx
            code = ((b1 & 0b00011111) << 6) | continuation_byte(i + 1)
            i += 2

        elif b1 & 0b11110000 == 0b11100000:
            # 1110xxxx 10xxxxxx 10xxxxxx
            code = ((b1 & 0b00001111) << 12) | (continuation_byte(i + 1) << 6) | continuation_byte(i + 2)
            i += 3

        elif b1 & 0b11111000 == 0b11110000:
            # 11110xxx 10xxxxxx 10xxxxxx 10xxxxxx
            code = ((b1 & 0b00000111) << 18) | (continuation_byte(i + 1) << 12) | (continuation_byte(i + 2) << 6) | continuation_byte(i + 3)
            i += 4

        else:
            raise ValueError(f"Invalid UTF-8 start byte at index {i}: {b1}")

        chars.append(chr(code))

    return "".join(chars)


custom_encode("rahul ਇ 👋"), custom_decode(custom_encode("rahul ਇ 👋"))


([114, 97, 104, 117, 108, 32, 224, 168, 135, 32, 240, 159, 145, 139],
 'rahul ਇ 👋')

In [38]:
custom_encode("rahul ਇ 👋") == encode("rahul ਇ 👋")

True

In [39]:
s = "rahul ਇ 👋"
custom_decode(custom_encode(s)) == s

True